In [ ]:
# prompt: Create a Python function that uses the National Weather Service API to retrieve the weather based on latitude and longitude. Use proper type hints and docstring comments as specified by the Python PEP 8 style guide.

import requests
from typing import Dict, Any

def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
  """
  Retrieves weather information from the National Weather Service API based on latitude and longitude.

  Args:
    latitude: The latitude of the location.
    longitude: The longitude of the location.

  Returns:
    A dictionary containing the weather information.
  """
  base_url = "https://api.weather.gov/points/"
  url = f"{base_url}{latitude},{longitude}"
  response = requests.get(url)
  response.raise_for_status()  # Raise an exception for bad status codes
  data = response.json()

  # The API returns a 'properties' dictionary which contains the forecast URL
  forecast_url = data.get("properties", {}).get("forecast")
  if not forecast_url:
    raise ValueError("Could not find forecast URL in API response.")

  forecast_response = requests.get(forecast_url)
  forecast_response.raise_for_status()
  return forecast_response.json()


In [ ]:
# prompt: Similarly, create a function that uses the Google Maps Geocoding API to convert places to latitude
# and longitude

import requests
from typing import Dict, Any

def get_coordinates_by_place(place: str) -> Dict[str, Any]:
  """
  Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.

  Args:
    place: The name of the place (e.g., "New York City").

  Returns:
    A dictionary containing the latitude and longitude.
  """
  # Replace with your actual Google Maps API key
  api_key = "xxx"
  base_url = "https://maps.googleapis.com/maps/api/geocode/json?"
  url = f"{base_url}address={place}&key={api_key}"
  response = requests.get(url)
  response.raise_for_status()
  data = response.json()

  if data.get("status") != "OK":
    raise ValueError(f"Google Maps API error: {data.get('status')}")

  location = data.get("results", [])[0].get("geometry", {}).get("location", {})
  return location


In [ ]:
import requests
from typing import Dict, Any

# prompt: Add the functions as tools for and agent using the ADK. Give the agent appropriate instructions.

!pip install autogen
from autogen import UserProxyAgent, AssistantAgent, Agent, GroupChat, GroupChatManager

# Initialize agents
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    is_termination_msg=lambda x: "TERMINATE" in x,
    code_execution_config={"last_n_messages": 2, "work_dir": "coding"},
    function_map={
        "get_weather_by_coordinates": get_weather_by_coordinates,
        "get_coordinates_by_place": get_coordinates_by_place,
    }
)

# Define the tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_by_coordinates",
            "description": "Retrieves weather information from the National Weather Service API based on latitude and longitude.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "The latitude of the location."
                    },
                    "longitude": {
                        "type": "number",
                        "description": "The longitude of the location."
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_coordinates_by_place",
            "description": "Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "place": {
                        "type": "string",
                        "description": "The name of the place (e.g., 'New York City')."
                    }
                },
                "required": ["place"]
            }
        }
    }
]

# Create an agent with the defined tools and instructions
weather_agent = AssistantAgent(
    name="weather_agent",
    llm_config={
        "tools": tools,
        "config_list": [
            {
                "model": MODEL_GEMINI_2_5_FLASH,
                "api_type": "google",
            }
        ],
    },
    system_message="You are a helpful assistant that can provide weather information. You can use the provided tools to get weather data by coordinates or get coordinates by place name. Always ask for clarification if you are unsure about the input."
)

# Create a group chat
groupchat = GroupChat(
    agents=[user_proxy, weather_agent],
    messages=[],
    max_round=10
)

# Create a group chat manager
manager = GroupChatManager(
    groupchat=groupchat,
    llm_config={
        "config_list": [
            {
                "model": MODEL_GEMINI_2_5_FLASH,
                "api_type": "google",
            }
        ]
    }
)

# Start the conversation
user_proxy.initiate_chat(
    manager,
    message="What is the weather in London?"
)


user_proxy (to chat_manager):

What is the weather in London?

--------------------------------------------------------------------------------



Next speaker: weather_agent

weather_agent (to chat_manager):


***** Suggested tool call (9073): get_coordinates_by_place *****
Arguments: 
{"place": "London"}
****************************************************************

--------------------------------------------------------------------------------



Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_coordinates_by_place...
Call ID: 9073
Input arguments: {'place': 'London'}

>>>>>>>> EXECUTED FUNCTION get_coordinates_by_place...
Call ID: 9073
Input arguments: {'place': 'London'}
Output:
{'lat': 51.5072178, 'lng': -0.1275862}
user_proxy (to chat_manager):

***** Response from calling tool (9073) *****
{'lat': 51.5072178, 'lng': -0.1275862}
*********************************************

--------------------------------------------------------------------------------



Next speaker: weather_agent

weather_agent (to chat_manager):


***** Suggested tool call (7817): get_weather_by_coordinates *****
Arguments: 
{"latitude": 51.5072178, "longitude": -0.1275862}
******************************************************************

--------------------------------------------------------------------------------



Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_weather_by_coordinates...
Call ID: 7817
Input arguments: {'latitude': 51.5072178, 'longitude': -0.1275862}

>>>>>>>> EXECUTED FUNCTION get_weather_by_coordinates...
Call ID: 7817
Input arguments: {'latitude': 51.5072178, 'longitude': -0.1275862}
Output:
Error: 404 Client Error: Not Found for url: https://api.weather.gov/points/51.5072,-0.1276
user_proxy (to chat_manager):

***** Response from calling tool (7817) *****
Error: 404 Client Error: Not Found for url: https://api.weather.gov/points/51.5072,-0.1276
*********************************************

--------------------------------------------------------------------------------


...


Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...
user_proxy (to chat_manager):



--------------------------------------------------------------------------------



Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...
user_proxy (to chat_manager):



--------------------------------------------------------------------------------



Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...
user_proxy (to chat_manager):



--------------------------------------------------------------------------------



Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...
user_proxy (to chat_manager):



--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (4e4684a5-0b0a-4d45-88fe-db8c97c7c001): Maximum rounds (10) reached


ChatResult(chat_id=322427079821035726403933753347646108473, chat_history=[{'content': 'What is the weather in London?', 'role': 'assistant', 'name': 'user_proxy'}, {'content': '', 'name': 'weather_agent', 'tool_calls': [{'id': '9073', 'function': {'arguments': '{"place": "London"}', 'name': 'get_coordinates_by_place'}, 'type': 'function'}], 'role': 'assistant'}, {'content': "{'lat': 51.5072178, 'lng': -0.1275862}", 'tool_responses': [{'tool_call_id': '9073', 'role': 'tool', 'content': "{'lat': 51.5072178, 'lng': -0.1275862}"}], 'role': 'tool', 'name': 'user_proxy'}, {'content': '', 'name': 'weather_agent', 'tool_calls': [{'id': '7817', 'function': {'arguments': '{"latitude": 51.5072178, "longitude": -0.1275862}', 'name': 'get_weather_by_coordinates'}, 'type': 'function'}], 'role': 'assistant'}, {'content': 'Error: 404 Client Error: Not Found for url: https://api.weather.gov/points/51.5072,-0.1276', 'tool_responses': [{'tool_call_id': '7817', 'role': 'tool', 'content': 'Error: 404 Clien

In [ ]:
# prompt: Add support for both Gemini and some other third-party model (Claude, GPT, etc.)

import requests
from typing import Dict, Any

def get_weather_by_coordinates(latitude: float, longitude: float) -> Dict[str, Any]:
  """
  Retrieves weather information from the National Weather Service API based on latitude and longitude.

  Args:
    latitude: The latitude of the location.
    longitude: The longitude of the location.

  Returns:
    A dictionary containing the weather information.
  """
  base_url = "https://api.weather.gov/points/"
  url = f"{base_url}{latitude},{longitude}"
  response = requests.get(url)
  response.raise_for_status()  # Raise an exception for bad status codes
  data = response.json()

  # The API returns a 'properties' dictionary which contains the forecast URL
  forecast_url = data.get("properties", {}).get("forecast")
  if not forecast_url:
    raise ValueError("Could not find forecast URL in API response.")

  forecast_response = requests.get(forecast_url)
  forecast_response.raise_for_status()
  return forecast_response.json()


import requests
from typing import Dict, Any

def get_coordinates_by_place(place: str) -> Dict[str, Any]:
  """
  Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.

  Args:
    place: The name of the place (e.g., "New York City").

  Returns:
    A dictionary containing the latitude and longitude.
  """
  # Replace with your actual Google Maps API key
  api_key = "xxx"
  base_url = "https://maps.googleapis.com/maps/api/geocode/json?"
  url = f"{base_url}address={place}&key={api_key}"
  response = requests.get(url)
  response.raise_for_status()
  data = response.json()

  if data.get("status") != "OK":
    raise ValueError(f"Google Maps API error: {data.get('status')}")

  location = data.get("results", [])[0].get("geometry", {}).get("location", {})
  return location

import requests
from typing import Dict, Any


!pip install autogen
from autogen import UserProxyAgent, AssistantAgent, Agent, GroupChat, GroupChatManager

# Initialize agents
user_proxy = UserProxyAgent(
    name="user_proxy",
    human_input_mode="TERMINATE",
    is_termination_msg=lambda x: "TERMINATE" in x,
    code_execution_config={"last_n_messages": 2, "work_dir": "coding"},
    function_map={
        "get_weather_by_coordinates": get_weather_by_coordinates,
        "get_coordinates_by_place": get_coordinates_by_place,
    }
)

# Define the tools
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather_by_coordinates",
            "description": "Retrieves weather information from the National Weather Service API based on latitude and longitude.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {
                        "type": "number",
                        "description": "The latitude of the location."
                    },
                    "longitude": {
                        "type": "number",
                        "description": "The longitude of the location."
                    }
                },
                "required": ["latitude", "longitude"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_coordinates_by_place",
            "description": "Retrieves latitude and longitude from the Google Maps Geocoding API based on a place name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "place": {
                        "type": "string",
                        "description": "The name of the place (e.g., 'New York City')."
                    }
                },
                "required": ["place"]
            }
        }
    }
]

# --- Model Configuration ---
# You can define multiple model configurations here.
# For Gemini, ensure you have the correct API key and

In [ ]:
# prompt: Write test code to demonstrate that the agent works for multiple US cities.

# Test with multiple US cities
cities = ["New York City", "Los Angeles", "Chicago", "Houston", "Phoenix"]

for city in cities:
    print(f"\n--- Testing weather for {city} ---")
    user_proxy.initiate_chat(
        manager,
        message=f"What is the weather in {city}?"
    )


--- Testing weather for New York City ---
user_proxy (to chat_manager):

What is the weather in New York City?

--------------------------------------------------------------------------------



Next speaker: weather_agent

weather_agent (to chat_manager):


***** Suggested tool call (7997): get_coordinates_by_place *****
Arguments: 
{"place": "New York City"}
****************************************************************

--------------------------------------------------------------------------------



Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_coordinates_by_place...
Call ID: 7997
Input arguments: {'place': 'New York City'}

>>>>>>>> EXECUTED FUNCTION get_coordinates_by_place...
Call ID: 7997
Input arguments: {'place': 'New York City'}
Output:
{'lat': 40.7127753, 'lng': -74.0059728}
user_proxy (to chat_manager):

***** Response from calling tool (7997) *****
{'lat': 40.7127753, 'lng': -74.0059728}
*********************************************

--------------------------------------------------------------------------------



Next speaker: weather_agent

weather_agent (to chat_manager):


***** Suggested tool call (4819): get_weather_by_coordinates *****
Arguments: 
{"latitude": 40.7127753, "longitude": -74.0059728}
******************************************************************

--------------------------------------------------------------------------------



Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_weather_by_coordinates...
Call ID: 4819
Input arguments: {'latitude': 40.7127753, 'longitude': -74.0059728}

>>>>>>>> EXECUTED FUNCTION get_weather_by_coordinates...
Call ID: 4819
Input arguments: {'latitude': 40.7127753, 'longitude': -74.0059728}
Output:
{'@context': ['https://geojson.org/geojson-ld/geojson-context.jsonld', {'@version': '1.1', 'wx': 'https://api.weather.gov/ontology#', 'geo': 'http://www.opengis.net/ont/geosparql#', 'unit': 'http://codes.wmo.int/common/unit/', '@vocab': 'https://api.weather.gov/ontology#'}], 'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-74.0009, 40.702], [-73.9965, 40.7237], [-74.0251, 40.727], [-74.0296, 40.7054], [-74.0009, 40.702]]]}, 'properties': {'units': 'us', 'forecastGenerator': 'BaselineForecastGenerator', 'generatedAt': '2026-03-02T19:22:03+00:00', 'updateTime': '2026-03-02T18:04:31+00:00', 'validTimes': '2026-03-02T12:00:00+00:

...


Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_coordinates_by_place...
Call ID: 9999
Input arguments: {'place': 'Chicago'}

>>>>>>>> EXECUTED FUNCTION get_coordinates_by_place...
Call ID: 9999
Input arguments: {'place': 'Chicago'}
Output:
{'lat': 41.88325, 'lng': -87.6323879}
user_proxy (to chat_manager):

***** Response from calling tool (9999) *****
{'lat': 41.88325, 'lng': -87.6323879}
*********************************************

--------------------------------------------------------------------------------



Next speaker: weather_agent

weather_agent (to chat_manager):


***** Suggested tool call (1544): get_weather_by_coordinates *****
Arguments: 
{"longitude": -87.6323879, "latitude": 41.88325}
******************************************************************

--------------------------------------------------------------------------------



Next speaker: user_proxy


>>>>>>>> USING AUTO REPLY...

>>>>>>>> EXECUTING FUNCTION get_weather_by_coordinates...
Call ID: 1544
Input arguments: {'longitude': -87.6323879, 'latitude': 41.88325}

>>>>>>>> EXECUTED FUNCTION get_weather_by_coordinates...
Call ID: 1544
Input arguments: {'longitude': -87.6323879, 'latitude': 41.88325}
Output:
{'@context': ['https://geojson.org/geojson-ld/geojson-context.jsonld', {'@version': '1.1', 'wx': 'https://api.weather.gov/ontology#', 'geo': 'http://www.opengis.net/ont/geosparql#', 'unit': 'http://codes.wmo.int/common/unit/', '@vocab': 'https://api.weather.gov/ontology#'}], 'type': 'Feature', 'geometry': {'type': 'Polygon', 'coordinates': [[[-87.6324, 41.8665], [-87.6308, 41.8883], [-87.66, 41.8895], [-87.6616, 41.8677], [-87.6324, 41.8665]]]}, 'properties': {'units': 'us', 'forecastGenerator': 'BaselineForecastGenerator', 'generatedAt': '2026-03-02T19:29:33+00:00', 'updateTime': '2026-03-02T19:11:19+00:00', 'validTimes': '2026-03-02T13:00:00+00:00/


Next speaker: weather_agent



KeyboardInterrupt: 